In [1]:
from spark_utils import SparkUtils

spark_url = "spark://spark-master:7077"
app_name = "Lab -3"

su=SparkUtils(spark_url,app_name)
su._spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 17:54:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
!pwd

/opt/spark/work-dir


In [3]:
import os

print(os.listdir("/opt/spark/work-dir/data"))


['qcommece']


In [24]:
"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating    
"""

column_types=[("Order_ID","long"),
              ("Company",'string'),
              ("City",'string'),
              ("Customer_Age",'int'),
              ("Order_Value",'float'),
              ("Delivery_Time_Min",'float'),
              ("Distance_Km",'float'),
              ("Items_Count",'float'),
              ("Product_Category",'string'),
              ("Payment_Method",'string'),
              ("Customer_Rating",'float'),
              ("Discount_Applied",'float'),
              ("Delivery_Partner_Rating",'float')   
]

qcommerce_schema = SparkUtils.generate_schema(column_types)

qcommerce_df = su._spark.read.option("header", "true").schema(qcommerce_schema).csv("/opt/spark/work-dir/data/qcommece")





qcommerce_df.printSchema()
qcommerce_df.show(5)


root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+----

In [25]:
#Create the new column with the conditions 
from pyspark.sql.functions import col, when,aggregate,avg,count

qcommerce_df2=qcommerce_df.withColumn("Delivery_SLA",when(col('Delivery_Time_Min')<=15,'Fast').when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON_TIME").when(col('Delivery_Time_Min')>20,'Late')).orderBy(col("Delivery_Time_Min").desc()).select("Order_ID", "Company", "City", 
                      "Delivery_Time_Min", "Delivery_SLA")
qcommerce_df2.show(5)

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1807126|Jio Mart|Haridwar|             40.0|        Late|
| 1529779|Jio Mart|Haridwar|             40.0|        Late|
| 1165764|Jio Mart|Haridwar|           39.994|        Late|
| 1729503|Jio Mart|Haridwar|           39.994|        Late|
| 1610720|Jio Mart|Haridwar|           39.994|        Late|
+--------+--------+--------+-----------------+------------+
only showing top 5 rows


In [26]:
qcommerce_df3=qcommerce_df.withColumn('Effective_Order_Value',col('Order_Value') * (1 -col('Discount_Applied')))

qcommerce_df3=qcommerce_df3.withColumn("Price_Bucket",
                                       when(col('Effective_Order_Value')<200,'Low')
                                       .when((col('Effective_Order_Value')>=200) & (col("Effective_Order_Value") <=500),"MEDIUM")
                                       .when(col('Effective_Order_Value')>500,'HIGH')
                                       )

result= qcommerce_df3.groupBy('Price_Bucket').agg(
    count('*').alias('Orders_count'),
    avg("Effective_Order_Value").alias('Average_Effective_Value')
).orderBy(col('Average_Effective_Value').desc())

result.show()


+------------+------------+-----------------------+
|Price_Bucket|Orders_count|Average_Effective_Value|
+------------+------------+-----------------------+
|        HIGH|      268953|      740.4337238893675|
|      MEDIUM|      210429|      358.0973233400432|
|         Low|      520618|     21.591204157544553|
+------------+------------+-----------------------+



In [27]:
qcommerce_df4=qcommerce_df.withColumn("Age_Group",
                                      when(col('Customer_Age')<25,'YOUNG')
                                      .when((col('Customer_Age')>=25) & (col('Customer_Age')<=44),'ADULT')
                                      .when(col('Customer_Age')>44,'SENIOR'))

qcommerce_df4_clean=qcommerce_df4.filter(
    (col('Customer_Age').isNotNull()) &
    (col('Customer_Age')>=0) &
    (col('Customer_Age')<=100)
)

result_df4=qcommerce_df4_clean.groupby('Age_Group','Product_Category').agg(
    count('*').alias('orders'),
    avg('Order_Value').alias('Average_Order_Value')
)

result_df4=result_df4.orderBy(
    col('Age_Group').asc(),
    col('Average_Order_Value').desc()
)

result_df4.show()



+---------+-------------------+------+-------------------+
|Age_Group|   Product_Category|orders|Average_Order_Value|
+---------+-------------------+------+-------------------+
|    ADULT|              Dairy| 68512|  573.4268899414931|
|    ADULT|          Household| 68110|  572.9259149188181|
|    ADULT|          Beverages| 67979|  572.0329877095578|
|    ADULT|          Groceries| 68291|  571.5250993844182|
|    ADULT|             Snacks| 68100|  570.3797974095505|
|    ADULT|Fruits & Vegetables| 67736|  569.8593599596651|
|    ADULT|      Personal Care| 68331|   569.512671998805|
|   SENIOR|              Dairy| 51025|   573.781117268945|
|   SENIOR|Fruits & Vegetables| 50642|  573.7244422534909|
|   SENIOR|             Snacks| 50924|   572.687851608881|
|   SENIOR|          Groceries| 51030|  572.2596391052539|
|   SENIOR|          Household| 50803|   571.172082385334|
|   SENIOR|      Personal Care| 50707|  571.1642560109325|
|   SENIOR|          Beverages| 50746|   568.14101323125

In [28]:
su._spark.stop()